# Simulations moteur & correcteurs

### Imports

In [1]:
import numpy as np
from control import tf, c2d, feedback, damp
from control_plotly import pzmap, step, impulse, bode, nichols, rlocus

### Constantes

In [13]:
### mt_ = moteur
mt_L = 1.4e-3 #H
mt_R = 1.4 #Ohm
mt_Km = 57e-3 #Nm/A
mt_J = 2.5e-5 #Kg/m2
mt_f = 8e-4 #N.m/(rad/s)

s = tf('s')

Mt_s = (mt_Km)/((mt_Km**2)+(mt_f+mt_J*s)*(mt_R+mt_L*s))
print(Mt_s)


               0.057
------------------------------------
3.5e-08 s^2 + 3.612e-05 s + 0.004369



### Equations

### Simulations

In [14]:
### Simulation moteur
Mt_s_rpm = (60/(2*np.pi))*Mt_s
tm = np.linspace(start=0,stop=0.1, num = 1000)
step(Mt_s_rpm, tm, x_title="Temps (s)", y_title ="RPM")

In [16]:

Mt_s_H = Mt_s_rpm/(1+Mt_s_rpm)
step([Mt_s_H*125,Mt_s_rpm],t=tm, x_title = "Temps (s)", y_title ="Commande")

### Discret

In [17]:
tm_z = np.arange(0, 1, 0.01)
Mt_z_rpm = c2d(Mt_s_rpm, 0.01, 'zoh')
print(Mt_z_rpm)
step([Mt_z_rpm,Mt_s_rpm], t=tm_z)


     88.12 z + 5.704
--------------------------
z^2 - 0.2469 z + 3.297e-05

dt = 0.01



In [18]:
Mt_z_H = Mt_z_rpm/(1+Mt_z_rpm)
step(Mt_z_H, t=tm_z)

In [19]:
offset = 10**(36.664/20) #+1 for non-oscillating graph
print(offset)
nichols([Mt_z_rpm,Mt_z_rpm/offset, Mt_z_rpm/80])

68.10829367676388


In [20]:
offset_z = 80
Mt_z_Hcorr = (Mt_z_rpm/(offset_z))/(1+(Mt_z_rpm/(offset_z)))
tm_z2 = np.arange(0, 1, 0.01)
step(Mt_z_Hcorr,t=tm_z2)

In [21]:
tm_z2 = np.arange(0, 5, 0.01)
step([Mt_z_Hcorr,Mt_s_H],t=tm_z2)

### Correcteur

In [57]:
Te = 0.07

Kp = 0.004#1/(offset + 120)
Ki = 0.008
Kd = -0.0001#0.09

z = tf('z')

print("Kp : ", Kp)
print("Ki : ", Ki)
print("Kd : ", Kd)

Cz = Kp+Ki*(z/(z-1))+Kd*((z-1)/z)

nichols([Cz*Mt_z_rpm, (Cz*Mt_z_rpm)/(1+Cz*Mt_z_rpm)])

Kp :  0.004
Ki :  0.008
Kd :  -0.0001


In [59]:
Mt_z_corr = (Cz*Mt_z_rpm)/(1+(Cz*Mt_z_rpm))
tsim = np.arange(0,2,Te)
step(1*Mt_z_corr*3000, tsim)

In [60]:
Mt_z_errcorr = 1/(1+(Cz*Mt_z_rpm))
step(Mt_z_errcorr*3800, tsim)

In [56]:
alpha_k = 100/(3800*(Kp+Kd+Ki))

step(1*Cz*Mt_z_errcorr*3800*alpha_k, t =np.arange(0,1,0.01), x_title="Temps (s)", y_title="Commande PWM (%)")